# Attention Mechanism

In [8]:
<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/01.webp?123" width="500px">

SyntaxError: invalid syntax (3791794997.py, line 1)

## Housekeeping

In [1]:
import sys
import os
import subprocess

print("Python executable:", sys.executable)
print("Python prefix:", sys.prefix)
print("Conda environment:", os.environ.get('CONDA_DEFAULT_ENV'))
print("Conda prefix:", os.environ.get('CONDA_PREFIX'))

# Check pip location
result = subprocess.run(['which', 'pip'], capture_output=True, text=True)
print("Pip location:", result.stdout.strip())

# Check pip version
result = subprocess.run(['pip', '--version'], capture_output=True, text=True)
print("Pip version:", result.stdout.strip())

# Add environment's bin directory to PATH
env_bin = os.path.dirname(sys.executable)
current_path = os.environ['PATH']
if env_bin not in current_path:
    os.environ['PATH'] = f"{env_bin}:{current_path}"

# Verify fix
result = subprocess.run(['which', 'pip'], capture_output=True, text=True)
print("Fixed pip location:", result.stdout.strip())

Python executable: /home/sagemaker-user/.conda/envs/venv-LLMsFromScratch/bin/python3
Python prefix: /home/sagemaker-user/.conda/envs/venv-LLMsFromScratch
Conda environment: base
Conda prefix: /opt/conda
Pip location: /opt/conda/bin/pip
Pip version: pip 25.1.1 from /opt/conda/lib/python3.12/site-packages/pip (python 3.12)
Fixed pip location: /home/sagemaker-user/.conda/envs/venv-LLMsFromScratch/bin/pip


## 3. Coding Attention Mechanism
**1. Simplified Self Attention Mechanism** - A simplified self-attention technique to introduce the broader idea

**2. Self Attention** - Self-Attention with trainable weight that forms the basis of the mechanism used in LLMs

**3. Causal Attention** - A type of self-attention used in LLMs that allow a model to consider only previous and current inputs in a sequence, ensuring tempora; order during the text geenration

**4. Multi Head Attention** - An extension of self-attention and causal attention that enables the model to simultaneously attend to information from different representation subspaces

## 3.1 The problem with modeling long sequences

## 3.2  Capturing data dependencies with attention mechanisms

## 3.3 Attending to different parts of the input with self attention

Input Embedding Vector of Tokens * Attention Weight to weigh the importance of input (Trained) -> The context vector is computed as a combination of all input 

In [31]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [32]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [33]:
inputs[1]

tensor([0.5500, 0.8700, 0.6600])

In [34]:
inputs[1][0]

tensor(0.5500)

In [35]:
# Let's day the query in this case is x2 and we want to calculate it's attention to every other token

# Saving our query
input_query = inputs[1]
input_query

tensor([0.5500, 0.8700, 0.6600])

In [36]:
# In order to calculate attention we need to multiply elements(embedded token vector values i.e 0.55, .87 .66 to the values 
# of embedding vector of other tokens ) LIKE FOR X2, X1 = .55*.43 + .87*.15 + .66*.89
.55*.43 + .87*.15 + .66*.89
# What we did here essentially calcualte the attetion of x2 wrt to x1 by multiplying the values of embedding of x2 with values
# of embeddings of x1 

0.9544

In [37]:
input_1 = inputs[0]

In [38]:
# We are doing dot product using torch function
torch.dot(input_query, input_1)

tensor(0.9544)

In [39]:
# printing the values at each index of embedding vector 
for ele in inputs[0]:
    print(ele)

tensor(0.4300)
tensor(0.1500)
tensor(0.8900)


In [40]:
# Here we are using enumerate to get the index and value of an embedding vector of token x1 at index o of inputs
for idx, ele in enumerate(inputs[0]):
    print(idx, ele)

0 tensor(0.4300)
1 tensor(0.1500)
2 tensor(0.8900)


In [49]:
# Let's use this to calulate the self - attention of x2 with input[0] at x1
result = 0.000000
input_query = inputs[1]


for idx, ele in enumerate(inputs[0]):
    print(inputs[0][idx], " * " ,input_query[idx], " = ", inputs[0][idx] * input_query[idx])
    result += inputs[0][idx] * input_query[idx]
    # print(results)
    
print(result)

tensor(0.4300)  *  tensor(0.5500)  =  tensor(0.2365)
tensor(0.1500)  *  tensor(0.8700)  =  tensor(0.1305)
tensor(0.8900)  *  tensor(0.6600)  =  tensor(0.5874)
tensor(0.9544)


In [51]:
# Let's make it easier
res = torch.dot(inputs[0], input_query)
res

tensor(0.9544)

In [62]:
# So far we have been canculating attention between x2(query) and x1(with token 1)
attn_scores_2 = torch.empty(inputs.shape[0])

for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, input_query)

print("Attention score for x2--> ", attn_scores_2)

Attention score for x2-->  tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [64]:
# We will normalize the values in this
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("Attention Weight:---> ", attn_weights_2_tmp)

Attention Weight:--->  tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])


In [59]:
attn_weights_2_tmp.sum()

tensor(1.0000)

In [60]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

softmax_naive(attn_scores_2)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [68]:
attn_weights_2 = torch.softmax(attn_scores_2, dim = 0)
attn_weights_2

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [71]:
# Computing the context vector with respect to token x2
query = inputs[1]

context_vector_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vector_2 += attn_weights_2[i] * x_i
print(context_vector_2)

tensor([0.4419, 0.6515, 0.5683])


## 3.3.1 A Simple Self Attention Mechanism without Trainable Weights

In [72]:
# Calculate the attetion vectors for all tokens of input sequence
attn_scores = torch.empty(6, 6)

for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i,j] = torch.dot(x_i, x_j)

print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [74]:
# Calcluating attention scores using matrix maultiplication
attn_scores = inputs @ inputs.T
attn_scores

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [76]:
# Calculating the attention weights (normalized attention scores)
attn_weights = torch.softmax(attn_scores, dim=1)
attn_weights

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [77]:
# Final Code
attn_scores = inputs @ inputs.T
attn_weight = torch.softmax(attn_scores, dim = 1)

# All Context Vectors
all_context_vecs = attn_weights @ inputs
all_context_vecs

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

## 3.4 Implementing Self-Attention with trainable weights

### 3.4.1 Computing the atttention weights step by step

In [78]:
# Defining Few Variables
x_2 = inputs[1] # The second input element
d_in = inputs.shape[1] # THe nput embedding size, d= 3
d_out = 2 # The output embedding size, d_out = 2

In [87]:
# Let's Initilize three weight matrix - These are different from attention weight matrix
torch.manual_seed(35)
# We are setting requires_grad = False to redcuce clutter in the outputs, 
# But if were to use this weighted matrix in model training we would set it to True
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad = False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad = False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad = False)

print(W_query)
print(W_key)
print(W_value)

Parameter containing:
tensor([[0.2621, 0.1475],
        [0.9375, 0.1900],
        [0.2607, 0.0034]])
Parameter containing:
tensor([[0.0207, 0.2002],
        [0.2121, 0.1974],
        [0.5047, 0.2951]])
Parameter containing:
tensor([[0.3679, 0.8430],
        [0.8577, 0.2623],
        [0.6367, 0.3923]])


In [89]:
# Here we are computing Q,K,V matrix for x2 using the query attention weight and values of other matrix. 
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

print(query_2)
print(key_2)
print(value_2)

tensor([1.1318, 0.2487])
tensor([0.5290, 0.4766])
tensor([1.3688, 0.9508])


```Weighted Parameters are the fundamental, learned coeffients that define the networks connections, while attention weights are dynamic, context-specific values```

In [93]:
keys = inputs @ W_key
values = inputs @ W_value

print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

print(keys)
print(values)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])
tensor([[0.4899, 0.3784],
        [0.5290, 0.4766],
        [0.5151, 0.4708],
        [0.2941, 0.2559],
        [0.1194, 0.2330],
        [0.4483, 0.3303]])
tensor([[0.8535, 0.7510],
        [1.3688, 0.9508],
        [1.3463, 0.9546],
        [0.7885, 0.4671],
        [0.5614, 0.7539],
        [1.0548, 0.4678]])


In [96]:
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2)

print(keys_2)
print(attn_score_22)

tensor([0.5290, 0.4766])
tensor(0.7173)


In [97]:
# Calculating all attention scores via matrix multiplication
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

tensor([0.6486, 0.7173, 0.7001, 0.3966, 0.1931, 0.5896])


In [98]:
d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim = -1)
print(attn_weights_2)

tensor([0.1783, 0.1872, 0.1849, 0.1492, 0.1292, 0.1710])


In [99]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([1.0281, 0.7356])


## 3.1 A compact self-attention class

In [102]:
import torch.nn as nn
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self,x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value

        attn_scores = queries @ keys.T

        attn_weights = torch.softmax( attn_scores / keys.shape[-1]**0.5 , dim =1)

        context_vec = attn_weights @ values

        return context_vec

sa_v1 = SelfAttention_v1(d_in, d_out)
sa_v1(inputs)

tensor([[1.1045, 0.7201],
        [1.1494, 0.7540],
        [1.1472, 0.7523],
        [1.0862, 0.7071],
        [1.0609, 0.6851],
        [1.1157, 0.7304]], grad_fn=<MmBackward0>)

In [106]:
# Better version

torch.manual_seed(35)

class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias = False):
        super().__init__()
        self.W_query = torch.nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_key = torch.nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_value = torch.nn.Linear(d_in, d_out, bias = qkv_bias)

    def forward(self, x):
        queries = self.W_query(inputs)
        keys = self.W_key(inputs)
        values = self.W_value(inputs)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / d_k**0.5, dim = -1)
        context_vec = attn_weights @ values

        return context_vec

sa_v2 = SelfAttention_v2(d_in, d_out)
sa_v2(inputs)

tensor([[ 0.3809, -0.0946],
        [ 0.3849, -0.0963],
        [ 0.3850, -0.0963],
        [ 0.3845, -0.0945],
        [ 0.3857, -0.0947],
        [ 0.3839, -0.0948]], grad_fn=<MmBackward0>)

## 3.5 Hiding Future words with casual attention

### 3.5.1 Applying a casual attention mask

In [116]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim = -1)
print(attn_weights)

tensor([[0.1750, 0.1658, 0.1661, 0.1625, 0.1704, 0.1601],
        [0.1742, 0.1777, 0.1779, 0.1517, 0.1667, 0.1519],
        [0.1741, 0.1778, 0.1780, 0.1516, 0.1666, 0.1520],
        [0.1705, 0.1734, 0.1735, 0.1579, 0.1664, 0.1583],
        [0.1697, 0.1765, 0.1766, 0.1554, 0.1652, 0.1566],
        [0.1721, 0.1728, 0.1729, 0.1576, 0.1672, 0.1574]],
       grad_fn=<SoftmaxBackward0>)


In [117]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(make_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [119]:
masked_simple = attn_weights * mask_simple
print(masked_simple)

tensor([[0.1750, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1742, 0.1777, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1741, 0.1778, 0.1780, 0.0000, 0.0000, 0.0000],
        [0.1705, 0.1734, 0.1735, 0.1579, 0.0000, 0.0000],
        [0.1697, 0.1765, 0.1766, 0.1554, 0.1652, 0.0000],
        [0.1721, 0.1728, 0.1729, 0.1576, 0.1672, 0.1574]],
       grad_fn=<MulBackward0>)


In [122]:
row_sums = masked_simple.sum(dim =-1, keepdim = True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4951, 0.5049, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3286, 0.3355, 0.3359, 0.0000, 0.0000, 0.0000],
        [0.2525, 0.2568, 0.2569, 0.2337, 0.0000, 0.0000],
        [0.2012, 0.2093, 0.2094, 0.1843, 0.1958, 0.0000],
        [0.1721, 0.1728, 0.1729, 0.1576, 0.1672, 0.1574]],
       grad_fn=<DivBackward0>)


In [123]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal = 1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[0.0956,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.3943, 0.4219,   -inf,   -inf,   -inf,   -inf],
        [0.3951, 0.4247, 0.4262,   -inf,   -inf,   -inf],
        [0.2302, 0.2542, 0.2549, 0.1211,   -inf,   -inf],
        [0.2986, 0.3549, 0.3552, 0.1746, 0.2606,   -inf],
        [0.2356, 0.2412, 0.2423, 0.1107, 0.1951, 0.1094]],
       grad_fn=<MaskedFillBackward0>)


In [124]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim =1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4951, 0.5049, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3286, 0.3355, 0.3359, 0.0000, 0.0000, 0.0000],
        [0.2525, 0.2568, 0.2569, 0.2337, 0.0000, 0.0000],
        [0.2012, 0.2093, 0.2094, 0.1843, 0.1958, 0.0000],
        [0.1721, 0.1728, 0.1729, 0.1576, 0.1672, 0.1574]],
       grad_fn=<SoftmaxBackward0>)


In [126]:
torch.manual_seed(35)
dropout = torch.nn.Dropout(0.5)
example = torch.ones(6,6)
print(dropout(example))

tensor([[0., 2., 2., 2., 2., 2.],
        [2., 0., 2., 0., 2., 2.],
        [2., 2., 2., 0., 2., 0.],
        [0., 2., 2., 0., 2., 0.],
        [0., 2., 2., 0., 0., 2.],
        [0., 0., 2., 2., 2., 2.]])


In [127]:
torch.manual_seed(35)
print(dropout(attn_weights))

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.9902, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6572, 0.6711, 0.6718, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.5136, 0.5139, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4187, 0.4188, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.3458, 0.3151, 0.3345, 0.3148]],
       grad_fn=<MulBackward0>)


### 3.5.3 Implementing a compact casual attention class

In [129]:
batch = torch.stack((inputs, inputs), dim =0)
print(batch.shape)

torch.Size([2, 6, 3])


### 3.3 A compact casual attention class

In [142]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias= False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias = qkv_bias)

        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens , d_in = x.shape # batch , 2, 6, 3
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2) 
        attn_scores.masked_fill_( self.mask.bool()[:num_tokens, :num_tokens], -torch.inf) # _ ops are in place  to avoid unnecessary memory copies
        attn_weights = torch.softmax( attn_scores / keys.shape[-1] ** 0.5, dim =-1)
        attn_weights = self.dropout(attn_weights)
        context_vec = attn_weights @ values
        return context_vec


context_length = batch.shape[1]
dropout = 0.0
ca = CausalAttention(d_in, d_out, context_length, dropout)
context_vecs = ca(batch)
print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([2, 6, 2])


## 3.6 Extensing Single Head Attention to Multi Head Attention
### 3.6.1 Stacking multiple single head attention layers

## Listing 3.4 A Wrapper Class to implement multi-head attention

In [151]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads=2, qkv_bias = False):
        super().__init__()
        self.heads = nn.ModuleList([CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) for _ in range(num_heads)])

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim =-1)
        

In [153]:
contxt_length = batch.shape[1]
d_in, d_out = 3, 2

mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, dropout = 0.0, num_heads=2)
mha(batch)

tensor([[[-0.4808,  0.1497,  0.0911,  0.3199],
         [-0.5885,  0.0495,  0.3120,  0.2668],
         [-0.6173,  0.0243,  0.3741,  0.2539],
         [-0.5528, -0.0010,  0.3431,  0.2152],
         [-0.5043,  0.0427,  0.3597,  0.2195],
         [-0.5017,  0.0014,  0.3434,  0.1939]],

        [[-0.4808,  0.1497,  0.0911,  0.3199],
         [-0.5885,  0.0495,  0.3120,  0.2668],
         [-0.6173,  0.0243,  0.3741,  0.2539],
         [-0.5528, -0.0010,  0.3431,  0.2152],
         [-0.5043,  0.0427,  0.3597,  0.2195],
         [-0.5017,  0.0014,  0.3434,  0.1939]]], grad_fn=<CatBackward0>)

## Listing 3.5 An efficient multi head attention class

Input (b, tokens, d_in)

  → Project to Q, K, V (b, tokens, d_out)
  
  → Split into heads (b, tokens, heads, head_dim)
  
  → Rearrange (b, heads, tokens, head_dim)
  
  → Q·Kᵀ → scores (b, heads, tokens, tokens)
  
  → Mask future + scale + softmax → weights
  
  → Weights · V → attended output (b, heads, tokens, head_dim)
  
  → Rearrange + concatenate → (b, tokens, d_out)
  
  → Output projection → final result (b, tokens, d_out)

In [175]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias = False):
        super().__init__()
        """The key operation is to split the d_out dimension into num_heads and head_dim, 
        where head_dim = d_out / num_heads. This splitting is then achieved using the .view method: 
        a tensor of dimensions (b, num_tokens, d_out) is reshaped to dimension (b, num_tokens, num_heads, head_dim)."""
        assert (d_out % num_heads == 0), "d_out must be dvisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads #1 Reduces the projection dim to match the desired output dim 

        self.W_query = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias = qkv_bias)

        # What: A final linear layer after all heads are concatenated.
        # Why: Each head works independently — this layer lets the model mix information across heads. Head 1's findings can interact with Head 3's findings here.
        self.out_proj = nn.Linear(d_out, d_out) #2 Uses a Linear layer to combine head outputs 

        # What: Creates a dropout layer (randomly zeros out elements during training).
        # Why: Prevents the model from relying too heavily on specific attention patterns → reduces overfitting.
        self.dropout = nn.Dropout(dropout)

        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal = 1))

    def forward(self, x):
        """
        What: Unpacks the input tensor's dimensions.
        Why: We need these values for reshaping later.
        
        b = batch size (how many sentences at once)
        num_tokens = sequence length (how many words/tokens)
        d_in = embedding dimension
        """
        b, num_tokens, d_in = x.shape

        """
        What: Multiplies input by the Q, K, V weight matrices. Each output shape: (b, num_tokens, d_out).
        
        Why: Transforms raw token embeddings into the three distinct roles needed for attention.
        """
        keys = self.W_key(x) #3 Tensor shape: (b, num_tokens, d_out) 
        queries = self.W_query(x)
        values = self.W_value(x)

        """
        What: Reshapes from (b, tokens, d_out) → (b, tokens, num_heads, head_dim).
        E.g., (2, 10, 512) → (2, 10, 8, 64)
        
        Why: Splits the single large vector into separate chunks for each head. Now each head has its own 64-dim slice to work with independently.
        """
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) #4 We implicitly split the matrix by adding a num_heads dimension. Then we unroll the last dim: (b, num_tokens, d_out) -&gt; (b, num_tokens, num_heads, head_dim). 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim) 

        """
        What: Swaps dimensions 1 and 2: (b, tokens, heads, head_dim) → (b, heads, tokens, head_dim).
        Why: We want each head to process ALL tokens. By putting heads in dimension 1, PyTorch's batch matrix multiplication treats each head as a separate "batch item" — they all compute in parallel automatically.
        """
        keys = keys.transpose(1,2) 
        queries = queries.transpose(1,2)
        values = values.transpose(1,2)


        """
        What: Matrix multiplication of queries with transposed keys.
        queries: (b, heads, tokens, head_dim) @ keys: (b, heads, head_dim, tokens)
        Result: (b, heads, tokens, tokens)
        Why: This computes the dot product between every query and every key. High dot product = "these two tokens are relevant to each other". The result is a tokens × tokens score matrix per head.
        """
        attn_scores = queries @ keys.transpose(2,3)

        # What: Takes only the top-left portion of the mask matching the actual sequence length, converted to boolean.
        # Why: The mask was created for context_length (e.g. 1024), but our current input might only have 10 tokens. We trim it to 10×10.
        mask_bool = self.mask.bool ()[:num_tokens, :num_tokens]

        # What: Everywhere the mask is True (= future positions), replace the score with -∞.
        # Why: When we apply softmax next, e^(-∞) = 0. So future tokens get zero attention weight — the model literally cannot look ahead. This makes it causal (autoregressive).
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        """
        What: Two things:

        Divide by √head_dim (e.g. √64 = 8)
        Apply softmax along the last dimension
        Why:
        
        Scaling: Without it, dot products grow large with high dimensions, making softmax output nearly one-hot (all attention on one token). Scaling keeps gradients healthy.
        Softmax: Converts raw scores into probabilities (0 to 1, summing to 1 per row). Now each token has a probability distribution over which other tokens to attend to.
        """
        attn_weights = torch.softmax(attn_scores/ keys.shape[-1] **0.5, dim =-1)

        # What: Randomly zeros out some attention weights during training. Why: Forces the model to not rely on just one or two tokens — makes it more robust.
        attn_weights = self.dropout(attn_weights)


        
        """
        What:
        attn_weights @ values — weighted sum of value vectors. Shape: (b, heads, tokens, head_dim)
        .transpose(1, 2) — back to (b, tokens, heads, head_dim)
        
        Why: Each token's output is now a blend of other tokens' values, weighted by relevance. We transpose to get tokens back in dimension 1 (preparing for concatenation).
        """
        context_vec = (attn_weights @ values).transpose(1,2)

        """
        What:
        .contiguous() — ensures memory layout is contiguous after transpose
        .view(b, tokens, d_out) — flattens (heads, head_dim) back into d_out
        E.g., (2, 10, 8, 64) → (2, 10, 512)
        
        Why: Concatenates all heads' outputs into a single vector. The 8 separate 64-dim results become one 512-dim result again.
        """
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)

        # What: Passes concatenated output through a final linear layer.
        # Why: Heads worked independently — this allows cross-head information mixing. Also adds an extra learnable transformation for richer representations.
        context_vec = self.out_proj(context_vec)

        return context_vec

In [176]:
a = torch.tensor([[[[0.2745, 0.6584, 0.2775, 0.8573],    #1
                    [0.8993, 0.0390, 0.9268, 0.7388],
                    [0.7179, 0.7058, 0.9156, 0.4340]],

                   [[0.0772, 0.3565, 0.1479, 0.5331],
                    [0.4066, 0.2318, 0.4545, 0.9737],
                    [0.4606, 0.5159, 0.4220, 0.5786]]]])

print(a @ a.transpose(2,3))

tensor([[[[1.3208, 1.1631, 1.2879],
          [1.1631, 2.2150, 1.8424],
          [1.2879, 1.8424, 2.0402]],

         [[0.4391, 0.7003, 0.5903],
          [0.7003, 1.3737, 1.0620],
          [0.5903, 1.0620, 0.9912]]]])


In [177]:
torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])
